In [ ]:
%pip -q install seaborn

Note: you may need to restart the kernel to use updated packages.


In [4]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load datasets
train_path = "C:\\Users\\Madis\\Downloads\\preprocessing\\train_cleaned.csv"
val_path = "C:\\Users\\Madis\\Downloads\\preprocessing\\validation_cleaned.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)

print("Training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)

Training dataset: (19300, 530)
Validation dataset: (1111, 530)


In [5]:
# Define the Zero-Rule Baseline Classifier directly
class ZeroRuleClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.classes_, counts = np.unique(y, return_counts=True)
        self.majority_class_ = self.classes_[np.argmax(counts)]
        return self
    def predict(self, X):
        return np.array([self.majority_class_] * len(X))
    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)



In [6]:
# Identify Wi-Fi RSSI features
wap_cols = [
    col for col in train_df.columns
    if col.startswith("WAP")
]

# Separate features and target
X_raw = train_df[wap_cols].copy()
y = train_df["LOCATION_CLASS"].copy()

X_validation_raw = val_df[wap_cols].copy()
y_validation = val_df["LOCATION_CLASS"].copy()

print("Number of WAP features:", len(wap_cols))
print("Number of location classes:", y.nunique())

# Check for conventional missing values
print("NaN values in training:", X_raw.isna().sum().sum())
print("NaN values in validation:", X_validation_raw.isna().sum().sum())

Number of WAP features: 520
Number of location classes: 13
NaN values in training: 0
NaN values in validation: 0


In [7]:
# CONFIGURATION A: Original RSSI encoding
X_original = X_raw.copy()
X_validation_original = X_validation_raw.copy()

# CONFIGURATION B: Replace undetected WAP values (100) with -100
X_minus100 = X_raw.replace(100, -100)
X_validation_minus100 = X_validation_raw.replace(100, -100)

# Store both configurations
configurations = {
    "Original (100)": (X_original, X_validation_original),
    "Modified (-100)": (X_minus100, X_validation_minus100)
}

# Verify replacement
print("Original encoding - number of 100 values:",
      (X_original == 100).sum().sum())

print("Modified encoding - number of 100 values:",
      (X_minus100 == 100).sum().sum())

print("Modified encoding - number of -100 values:",
      (X_minus100 == -100).sum().sum())

Original encoding - number of 100 values: 0
Modified encoding - number of 100 values: 0
Modified encoding - number of -100 values: 239


In [8]:
# Create sample indices
indices = np.arange(len(X_raw))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Split target labels
y_train = y.iloc[train_indices]
y_test = y.iloc[test_indices]

print("Training samples:", len(train_indices))
print("Testing samples:", len(test_indices))

Training samples: 15440
Testing samples: 3860
